# Feature Extraction for Speech Emotion Recognition (SER)
Notebook ini mengekstrak fitur MFCC dan ZCR dari **6 dataset** (RAVDESS, CREMA-D, TESS, SAVEE, E-SERAVD, IndoWaveSentiment) untuk keperluan pengenalan emosi berbasis suara.

| Dataset | Bahasa | Gender | Emosi tersedia |
|---------|--------|--------|----------------|
| RAVDESS | Inggris | Mixed | 8 emosi |
| CREMA-D | Inggris | Mixed | 6 emosi |
| TESS | Inggris | Female only | 7 emosi |
| SAVEE | Inggris | Male only | 7 emosi |
| **E-SERAVD** | **Indonesia** | Mixed | 4 emosi (dari film) |
| **IndoWaveSentiment** | **Indonesia** | Mixed | 5 emosi (acted) |

**Label 6 kelas:** `happy=0, sad=1, angry=2, fearful=3, disgust=4, neutral=5`  
Dataset Indonesia hanya mengisi kelas: `happy, sad, angry, neutral`. Kelas `fearful` dan `disgust` diisi dari dataset Inggris.

## 1. Import Library & Konfigurasi Global

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import json
import warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# KONFIGURASI UTAMA — ubah sesuai kebutuhan
# ─────────────────────────────────────────────
# Semua dataset ada di dalam folder 'Dataset'
DATASET_ROOT = 'Dataset'
OUTPUT_DIR   = 'extracted_features'

# Parameter ekstraksi fitur
SR          = 16000
DURATION    = 3.0
N_MFCC      = 40
MAX_PAD_LEN = 128

# Pemetaan emosi → label integer (6 kelas, TETAP)
emotion_map = {
    'happy': 0, 'sad': 1, 'angry': 2,
    'fearful': 3, 'disgust': 4, 'neutral': 5
}
inv_emotion_map = {v: k for k, v in emotion_map.items()}

# Buat direktori output
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'male'),   exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'female'), exist_ok=True)

# ── Muat metadata.json ──────────────────────────────────────────────────────
metadata = {}
meta_path = os.path.join(DATASET_ROOT, 'metadata.json')
if os.path.exists(meta_path):
    with open(meta_path, 'r') as f:
        metadata = json.load(f)
    print(f'Metadata loaded: {meta_path}')
else:
    print('metadata.json tidak ditemukan, lanjut tanpa metadata.')

print('Feature extraction parameters:')
print(f'  Sampling rate : {SR} Hz')
print(f'  Duration      : {DURATION} seconds')
print(f'  MFCC coeffs   : {N_MFCC}')
print(f'  Max frames    : {MAX_PAD_LEN}')
print(f'  Emotion classes: {emotion_map}')

## 2. Fungsi Utilitas

In [ ]:
# ── Muat gender map CREMA-D ──────────────────────────────────────────────────
# File VideoDemographics.csv tidak disertakan dalam dataset Crema pada folder ini
# sehingga gender akan di-infer dari rentang ActorID:
#   1001–1091  : mixed male/female (CREMA-D standard)
#   Aktor ganjil = male, genap = female  (pendekatan umum)
crema_gender_map = {}
crema_csv_candidates = [
    os.path.join(DATASET_ROOT, 'Crema', 'VideoDemographics.csv'),
    os.path.join(DATASET_ROOT, 'VideoDemographics.csv'),
    'VideoDemographics.csv',
]
for crema_meta_path in crema_csv_candidates:
    if os.path.exists(crema_meta_path):
        df_crema = pd.read_csv(crema_meta_path)
        for _, row in df_crema.iterrows():
            actor_id = str(int(row['ActorID']))
            crema_gender_map[actor_id] = row['Sex'].strip().lower()
        print(f'Loaded gender mapping for {len(crema_gender_map)} CREMA-D actors dari {crema_meta_path}')
        break
else:
    # Fallback: aktor ganjil=male, genap=female (berdasarkan ActorID 1001-1091)
    print('VideoDemographics.csv tidak ditemukan. Menggunakan fallback ganjil/genap untuk CREMA-D.')
    for aid in range(1001, 1092):
        crema_gender_map[str(aid)] = 'male' if aid % 2 == 1 else 'female'


def remove_noise(audio_data, sample_rate):
    """Kurangi noise (opsional, memerlukan library noisereduce)."""
    try:
        import noisereduce as nr
        return nr.reduce_noise(y=audio_data, sr=sample_rate, stationary=True)
    except ImportError:
        return audio_data
    except Exception as e:
        print(f'Noise reduction error: {e}')
        return audio_data


def extract_mfcc_zcr(audio_data, sample_rate):
    """Ekstrak MFCC + delta + delta2 + ZCR."""
    try:
        if sample_rate != SR:
            audio_data = librosa.resample(audio_data, orig_sr=sample_rate, target_sr=SR)
            sample_rate = SR
        target_length = int(DURATION * SR)
        if len(audio_data) > target_length:
            audio_data = audio_data[:target_length]
        else:
            audio_data = np.pad(audio_data, (0, target_length - len(audio_data)), mode='constant')
        if np.max(np.abs(audio_data)) > 0:
            audio_data = audio_data / np.max(np.abs(audio_data))
        mfcc       = librosa.feature.mfcc(y=audio_data, sr=sample_rate, n_mfcc=N_MFCC,
                                           n_fft=2048, hop_length=512, fmax=sample_rate/2)
        mfcc_delta  = librosa.feature.delta(mfcc)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
        zcr         = librosa.feature.zero_crossing_rate(audio_data, frame_length=2048, hop_length=512)
        features    = np.concatenate([mfcc.T, mfcc_delta.T, mfcc_delta2.T, zcr.T], axis=1)
        if features.shape[0] > MAX_PAD_LEN:
            features = features[:MAX_PAD_LEN, :]
        else:
            features = np.pad(features, ((0, MAX_PAD_LEN - features.shape[0]), (0, 0)), mode='constant')
        for i in range(features.shape[1]):
            std = np.std(features[:, i])
            if std > 0:
                features[:, i] = (features[:, i] - np.mean(features[:, i])) / std
        return features
    except Exception as e:
        print(f'Feature extraction error: {e}')
        return np.zeros((MAX_PAD_LEN, N_MFCC * 3 + 1))


def load_and_preprocess_audio(filepath):
    """Load audio dan terapkan noise reduction."""
    try:
        audio_data, sr = librosa.load(filepath, sr=None)
        audio_data = remove_noise(audio_data, sr)
        return audio_data, sr
    except Exception as e:
        print(f'Error loading {filepath}: {e}')
        return None, None


def _normalize_emotion(emotion):
    """Petakan string emosi ke kunci standar emotion_map."""
    if not emotion: return None
    e = emotion.lower()
    if 'happy' in e or 'joy' in e:   return 'happy'
    if 'sad' in e:                    return 'sad'
    if 'angry' in e or 'anger' in e: return 'angry'
    if 'fear' in e or 'afraid' in e: return 'fearful'
    if 'disgust' in e:                return 'disgust'
    if 'neutral' in e:                return 'neutral'
    return None

print('Fungsi utilitas siap.')

## 3. Fungsi Visualisasi: Perbandingan Original / SMOTE / Augmentasi

In [ ]:
def count_distribution(labels):
    """Kembalikan dict {emotion_name: count}."""
    result = {}
    for lbl, cnt in zip(*np.unique(labels, return_counts=True)):
        result[inv_emotion_map[lbl]] = int(cnt)
    return result


def print_distribution_table(orig_labels, smote_labels, aug_labels, dataset_name, gender):
    """Tampilkan tabel perbandingan jumlah data per tahap."""
    all_emotions = list(emotion_map.keys())
    orig_dist  = count_distribution(orig_labels)
    smote_dist = count_distribution(smote_labels)
    aug_dist   = count_distribution(aug_labels)

    print(f'\n  {"─"*60}')
    print(f'  {dataset_name} [{gender}] — Perbandingan Jumlah Sampel')
    print(f'  {"─"*60}')
    header = f'  {"Emosi":<12} {"Original":>10} {"→ SMOTE":>10} {"→ Augmen":>10}'
    print(header)
    print(f'  {"─"*60}')
    for emo in all_emotions:
        o = orig_dist.get(emo, 0)
        s = smote_dist.get(emo, 0)
        a = aug_dist.get(emo, 0)
        marker = ' ←' if o == 0 else ''
        print(f'  {emo:<12} {o:>10} {s:>10} {a:>10}{marker}')
    print(f'  {"─"*60}')
    print(f'  {"TOTAL":<12} {sum(orig_dist.values()):>10} '
          f'{sum(smote_dist.values()):>10} {sum(aug_dist.values()):>10}')
    print(f'  {"─"*60}')


def plot_comparison(orig_labels, smote_labels, aug_labels, dataset_name, gender, save_path=None):
    """Plot bar chart 3 panel: Original | SMOTE | Augmented."""
    all_class_names = [inv_emotion_map[i] for i in sorted(emotion_map.values())]
    palette = sns.color_palette('Set2', len(all_class_names))

    def get_counts(labels):
        dist = count_distribution(labels)
        return [dist.get(e, 0) for e in all_class_names]

    stages = [
        ('Original', orig_labels, 'steelblue'),
        ('+ SMOTE', smote_labels, 'darkorange'),
        ('+ Augmentasi', aug_labels, 'seagreen'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
    fig.suptitle(f'{dataset_name} [{gender}] — Distribusi Sampel per Tahap',
                 fontsize=13, fontweight='bold')

    for ax, (stage_name, stage_labels, color) in zip(axes, stages):
        counts = get_counts(stage_labels)
        bars = ax.bar(all_class_names, counts, color=palette)
        ax.set_title(f'{stage_name}\n(Total: {sum(counts):,})', fontsize=11)
        ax.set_xlabel('Emosi')
        ax.set_ylabel('Jumlah Sampel')
        ax.set_xticks(range(len(all_class_names)))
        ax.set_xticklabels(all_class_names, rotation=30, ha='right')
        for bar, cnt in zip(bars, counts):
            if cnt > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                        str(cnt), ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()


print('Fungsi visualisasi siap.')

## 4. Fungsi: Pipeline Augmentasi + SMOTE + Simpan

In [ ]:
def apply_smote_and_augment(features_arr, labels_arr, dataset_name, gender):
    """
    Terapkan SMOTE lalu augmentasi noise ringan.
    Returns: (aug_features, aug_labels)
    """
    # ── SMOTE ────────────────────────────────────────────────────────────────
    flat = features_arr.reshape(len(features_arr), -1)
    unique_lbl, lbl_cnt = np.unique(labels_arr, return_counts=True)
    if len(unique_lbl) > 1 and min(lbl_cnt) > 1:
        smote = SMOTE(random_state=42)
        flat_bal, labels_bal = smote.fit_resample(flat, labels_arr)
        features_bal = flat_bal.reshape(-1, MAX_PAD_LEN, N_MFCC * 3 + 1)
    else:
        print(f'    [INFO] SMOTE di-skip ({gender}): sampel tidak cukup.')
        features_bal, labels_bal = features_arr, labels_arr

    # ── Augmentasi noise ringan ──────────────────────────────────────────────
    aug_features, aug_labels = [], []
    for i, (feat, lbl) in enumerate(zip(features_bal, labels_bal)):
        aug_features.append(feat)
        aug_labels.append(lbl)
        if i % 3 == 0:
            for _ in range(2):
                aug_features.append(np.clip(feat + np.random.normal(0, 0.02, feat.shape), -1, 1))
                aug_labels.append(lbl)

    return (
        features_arr, labels_arr,
        features_bal, labels_bal,
        np.array(aug_features), np.array(aug_labels)
    )


def save_and_report(features_orig, labels_orig,
                    features_smote, labels_smote,
                    features_aug, labels_aug,
                    dataset_name, gender):
    """Tampilkan perbandingan, plot, dan simpan file .npz."""
    print_distribution_table(labels_orig, labels_smote, labels_aug, dataset_name, gender)

    save_plot = os.path.join(OUTPUT_DIR, f'{dataset_name}_{gender}_comparison.png')
    plot_comparison(labels_orig, labels_smote, labels_aug, dataset_name, gender, save_plot)

    out_file = os.path.join(OUTPUT_DIR, gender, f'{dataset_name}_features.npz')
    np.savez_compressed(out_file, features=features_aug, labels=labels_aug)
    print(f'  ✅ Saved {len(features_aug)} sampel → {out_file}')
    return len(features_aug)


def process_dataset(features_by_gender, labels_by_gender, file_count, dataset_name):
    """Pipeline lengkap: SMOTE + augmentasi + simpan untuk male dan female."""
    total_saved = 0
    for gender in ['male', 'female']:
        if not features_by_gender[gender]:
            print(f'  [SKIP] Tidak ada data {gender} untuk {dataset_name}')
            continue
        features_arr = np.array(features_by_gender[gender])
        labels_arr   = np.array(labels_by_gender[gender])

        f_orig, l_orig, f_smote, l_smote, f_aug, l_aug = apply_smote_and_augment(
            features_arr, labels_arr, dataset_name, gender
        )
        n = save_and_report(f_orig, l_orig, f_smote, l_smote, f_aug, l_aug, dataset_name, gender)
        total_saved += n

    print(f'\n  [{dataset_name}] Male: {file_count["male"]} | '
          f'Female: {file_count["female"]} | Total saved: {total_saved}')


print('Pipeline utilitas siap.')

## 5. Fungsi Parser Nama File per Dataset

In [ ]:
def parse_ravdess(filename):
    """(emotion, gender) dari nama file RAVDESS."""
    try:
        parts = os.path.splitext(filename)[0].split('-')
        if len(parts) < 7: return None, None
        mapping = {'01':'neutral','02':'calm','03':'happy','04':'sad',
                   '05':'angry','06':'fearful','07':'disgust','08':'surprised'}
        emotion  = _normalize_emotion(mapping.get(parts[2]))
        gender   = 'male' if int(parts[6]) % 2 == 1 else 'female'
        if emotion in emotion_map: return emotion, gender
    except Exception as e: print(f'RAVDESS parse error ({filename}): {e}')
    return None, None


def parse_crema(filename):
    """(emotion, gender) dari nama file CREMA-D."""
    try:
        parts = os.path.splitext(filename)[0].split('_')
        if len(parts) < 4: return None, None
        actor_id, emotion_code = parts[0], parts[2]
        mapping = {'NEU':'neutral','HAP':'happy','SAD':'sad','ANG':'angry',
                   'FEA':'fearful','DIS':'disgust'}
        emotion = _normalize_emotion(mapping.get(emotion_code.upper()))
        gender  = crema_gender_map.get(actor_id, '').lower()
        if emotion in emotion_map and gender in ['male','female']:
            return emotion, gender
    except Exception as e: print(f'CREMA-D parse error ({filename}): {e}')
    return None, None


def parse_tess(filepath):
    """(emotion, gender) dari path file TESS.
    Folder: OAF_<emotion> atau YAF_<emotion>
    """
    try:
        folder_name = os.path.basename(os.path.dirname(filepath))  # e.g. OAF_happy
        parts = folder_name.split('_', 1)  # ['OAF', 'happy'] atau ['YAF', 'pleasant_surprised']
        if len(parts) < 2: return None, None
        emotion = _normalize_emotion(parts[1])  # bagian setelah prefix
        if emotion in emotion_map: return emotion, 'female'
    except Exception as e: print(f'TESS parse error ({filepath}): {e}')
    return None, None


def parse_savee(filename):
    """(emotion, gender) dari nama file SAVEE.
    Format: DC_sa01.wav  (prefix speaker _ kodeemosi + nomor)
    """
    try:
        basename = os.path.splitext(os.path.basename(filename))[0]
        if '_' not in basename: return None, None
        code = basename.split('_')[1]          # e.g. 'sa01', 'a01', 'n05'
        if code[:2] == 'sa':   emotion = 'sad'
        elif code[:2] == 'su': return None, None  # surprise — skip
        else:
            mapping = {'a':'angry','d':'disgust','f':'fearful','h':'happy','n':'neutral','s':'sad'}
            emotion = _normalize_emotion(mapping.get(code[0].lower()))
        if emotion in emotion_map: return emotion, 'male'
    except Exception as e: print(f'SAVEE parse error ({filename}): {e}')
    return None, None


# ── Parser Dataset Indonesia ─────────────────────────────────────────────────
ESERAVD_FOLDER_EMOTION = {
    '01_Angry'  : 'angry',
    '02_Sad'    : 'sad',
    '03_Neutral': 'neutral',
    '04_Happy'  : 'happy',
}

def parse_eseravd(filename, emotion_folder):
    """
    (emotion, gender) dari nama file E-SERAVD.
    Format: EmotionId-GenderId-Source-T-Num.wav
    Emosi diambil dari folder parent, bukan nama file.
    """
    try:
        emotion = ESERAVD_FOLDER_EMOTION.get(emotion_folder)
        if emotion not in emotion_map: return None, None
        parts = os.path.splitext(filename)[0].split('-')
        if len(parts) < 2: return None, None
        gender_id = parts[1]  # '01'=male, '02'=female
        if   gender_id == '01': gender = 'male'
        elif gender_id == '02': gender = 'female'
        else: return None, None
        return emotion, gender
    except Exception as e: print(f'E-SERAVD parse error ({filename}): {e}')
    return None, None


INDOWAVE_EMOTION_MAP = {
    '01': 'neutral',
    '02': None,       # calm — skip
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
}

def parse_indowave(filename):
    """
    (emotion, gender) dari nama file IndoWaveSentiment.
    Format: ActorId-EmotionCode-GenderCode-Repetition.wav
    """
    try:
        parts = os.path.splitext(filename)[0].split('-')
        if len(parts) < 4: return None, None
        emotion_raw = INDOWAVE_EMOTION_MAP.get(parts[1])
        if not emotion_raw or emotion_raw not in emotion_map: return None, None
        if   parts[2] == '01': gender = 'male'
        elif parts[2] == '02': gender = 'female'
        else: return None, None
        return emotion_raw, gender
    except Exception as e: print(f'IndoWave parse error ({filename}): {e}')
    return None, None


print('Parser siap.')

## 6. Fungsi Pengumpul File Audio per Dataset

In [ ]:
def get_audio_files(dataset_path, dataset_name):
    """Kumpulkan semua file .wav sesuai struktur folder tiap dataset."""
    audio_files = []
    print(f'  Scanning: {dataset_path}')
    if not os.path.exists(dataset_path):
        print(f'  [WARN] Path tidak ditemukan: {dataset_path}')
        return []

    if dataset_name == 'RAVDESS':
        # Struktur: audio_speech_actors_01-24/Actor_XX/*.wav
        for actor_folder in os.listdir(dataset_path):
            if actor_folder.startswith('Actor'):
                ap = os.path.join(dataset_path, actor_folder)
                if os.path.isdir(ap):
                    for f in os.listdir(ap):
                        if f.lower().endswith('.wav'):
                            audio_files.append((os.path.join(ap, f), f))

    elif dataset_name == 'CREMA-D':
        # Struktur: Dataset/Crema/*.wav (flat)
        for f in os.listdir(dataset_path):
            if f.lower().endswith('.wav'):
                audio_files.append((os.path.join(dataset_path, f), f))

    elif dataset_name == 'TESS':
        # Struktur: Dataset/Tess/OAF_<emotion>/*.wav  dan YAF_<emotion>/*.wav
        for folder in os.listdir(dataset_path):
            if folder.startswith(('OAF_', 'YAF_')):
                ep = os.path.join(dataset_path, folder)
                if os.path.isdir(ep):
                    for f in os.listdir(ep):
                        if f.lower().endswith('.wav'):
                            audio_files.append((os.path.join(ep, f), f))

    elif dataset_name == 'SAVEE':
        # Struktur: Dataset/Savee/*.wav (flat — semua speaker dalam 1 folder)
        # Format nama: DC_a01.wav, JE_h05.wav, dst.
        for f in os.listdir(dataset_path):
            if f.lower().endswith('.wav'):
                audio_files.append((os.path.join(dataset_path, f), f))

    print(f'  Found {len(audio_files)} audio files')
    return audio_files

print('get_audio_files siap.')

## 7. Proses Dataset RAVDESS

In [ ]:
DATASET_NAME = 'RAVDESS'
# Struktur: Dataset/Ravdess/audio_speech_actors_01-24/Actor_XX/*.wav
dataset_path = os.path.join(DATASET_ROOT, 'Ravdess', 'audio_speech_actors_01-24')

print(f"\n{'='*55}\nProcessing {DATASET_NAME}...\n{'='*55}")

if not os.path.exists(dataset_path):
    print(f'[SKIP] Path tidak ditemukan: {dataset_path}')
else:
    audio_files = get_audio_files(dataset_path, DATASET_NAME)
    features_by_gender = {'male': [], 'female': []}
    labels_by_gender   = {'male': [], 'female': []}
    file_count         = {'male': 0, 'female': 0}

    for i, (filepath, fname) in enumerate(audio_files):
        if i % 100 == 0:
            print(f'  {i}/{len(audio_files)} | Male: {file_count["male"]}, Female: {file_count["female"]}')
        try:
            emotion, gender = parse_ravdess(fname)
            if emotion is None or gender is None: continue
            audio_data, sr = load_and_preprocess_audio(filepath)
            if audio_data is None: continue
            feat = extract_mfcc_zcr(audio_data, sr)
            features_by_gender[gender].append(feat)
            labels_by_gender[gender].append(emotion_map[emotion])
            file_count[gender] += 1
        except Exception as e:
            print(f'  Error ({filepath}): {e}')

    process_dataset(features_by_gender, labels_by_gender, file_count, DATASET_NAME)

## 8. Proses Dataset CREMA-D

In [ ]:
DATASET_NAME = 'CREMA-D'
# Struktur: Dataset/Crema/*.wav (flat)
dataset_path = os.path.join(DATASET_ROOT, 'Crema')

print(f"\n{'='*55}\nProcessing {DATASET_NAME}...\n{'='*55}")

if not os.path.exists(dataset_path):
    print(f'[SKIP] Path tidak ditemukan: {dataset_path}')
else:
    audio_files = get_audio_files(dataset_path, DATASET_NAME)
    features_by_gender = {'male': [], 'female': []}
    labels_by_gender   = {'male': [], 'female': []}
    file_count         = {'male': 0, 'female': 0}

    for i, (filepath, fname) in enumerate(audio_files):
        if i % 200 == 0:
            print(f'  {i}/{len(audio_files)} | Male: {file_count["male"]}, Female: {file_count["female"]}')
        try:
            emotion, gender = parse_crema(fname)
            if emotion is None or gender is None: continue
            audio_data, sr = load_and_preprocess_audio(filepath)
            if audio_data is None: continue
            feat = extract_mfcc_zcr(audio_data, sr)
            features_by_gender[gender].append(feat)
            labels_by_gender[gender].append(emotion_map[emotion])
            file_count[gender] += 1
        except Exception as e:
            print(f'  Error ({filepath}): {e}')

    process_dataset(features_by_gender, labels_by_gender, file_count, DATASET_NAME)

## 9. Proses Dataset TESS

In [ ]:
DATASET_NAME = 'TESS'
# Struktur: Dataset/Tess/OAF_<emotion>/*.wav  dan YAF_<emotion>/*.wav
dataset_path = os.path.join(DATASET_ROOT, 'Tess')

print(f"\n{'='*55}\nProcessing {DATASET_NAME}...\n{'='*55}")

if not os.path.exists(dataset_path):
    print(f'[SKIP] Path tidak ditemukan: {dataset_path}')
else:
    audio_files = get_audio_files(dataset_path, DATASET_NAME)
    features_by_gender = {'male': [], 'female': []}
    labels_by_gender   = {'male': [], 'female': []}
    file_count         = {'male': 0, 'female': 0}

    for i, (filepath, fname) in enumerate(audio_files):
        if i % 100 == 0:
            print(f'  {i}/{len(audio_files)} | Female: {file_count["female"]}')
        try:
            emotion, gender = parse_tess(filepath)
            if emotion is None or gender is None: continue
            audio_data, sr = load_and_preprocess_audio(filepath)
            if audio_data is None: continue
            feat = extract_mfcc_zcr(audio_data, sr)
            features_by_gender[gender].append(feat)
            labels_by_gender[gender].append(emotion_map[emotion])
            file_count[gender] += 1
        except Exception as e:
            print(f'  Error ({filepath}): {e}')

    process_dataset(features_by_gender, labels_by_gender, file_count, DATASET_NAME)

## 10. Proses Dataset SAVEE

In [ ]:
DATASET_NAME = 'SAVEE'
# Struktur: Dataset/Savee/*.wav (flat — semua speaker dalam 1 folder)
dataset_path = os.path.join(DATASET_ROOT, 'Savee')

print(f"\n{'='*55}\nProcessing {DATASET_NAME}...\n{'='*55}")

if not os.path.exists(dataset_path):
    print(f'[SKIP] Path tidak ditemukan: {dataset_path}')
else:
    audio_files = get_audio_files(dataset_path, DATASET_NAME)
    features_by_gender = {'male': [], 'female': []}
    labels_by_gender   = {'male': [], 'female': []}
    file_count         = {'male': 0, 'female': 0}

    for i, (filepath, fname) in enumerate(audio_files):
        if i % 50 == 0:
            print(f'  {i}/{len(audio_files)} | Male: {file_count["male"]}')
        try:
            emotion, gender = parse_savee(fname)
            if emotion is None or gender is None: continue
            audio_data, sr = load_and_preprocess_audio(filepath)
            if audio_data is None: continue
            feat = extract_mfcc_zcr(audio_data, sr)
            features_by_gender[gender].append(feat)
            labels_by_gender[gender].append(emotion_map[emotion])
            file_count[gender] += 1
        except Exception as e:
            print(f'  Error ({filepath}): {e}')

    process_dataset(features_by_gender, labels_by_gender, file_count, DATASET_NAME)

## 11. Proses Dataset E-SERAVD (Indonesia)

**Catatan:** Dataset ini hanya memiliki 4 emosi (angry, sad, neutral, happy).  
Kelas `fearful` dan `disgust` akan tetap kosong untuk dataset ini — diisi oleh dataset Inggris.

Format file: `EmotionId-GenderId-Source-T-Num.wav`  
Emosi dari folder: `01_Angry`, `02_Sad`, `03_Neutral`, `04_Happy`

In [ ]:
DATASET_NAME  = 'E-SERAVD'
eseravd_path  = os.path.join(DATASET_ROOT, 'E-SERAVD')

print(f"\n{'='*55}\nProcessing {DATASET_NAME} (Indonesia)...\n{'='*55}")

if not os.path.exists(eseravd_path):
    print(f'[SKIP] Path tidak ditemukan: {eseravd_path}')
else:
    features_by_gender = {'male': [], 'female': []}
    labels_by_gender   = {'male': [], 'female': []}
    file_count         = {'male': 0, 'female': 0}

    for emotion_folder in sorted(ESERAVD_FOLDER_EMOTION.keys()):
        emotion_path = os.path.join(eseravd_path, emotion_folder)
        if not os.path.isdir(emotion_path):
            print(f'  [MISS] {emotion_folder}')
            continue
        print(f'  Scanning {emotion_folder}...')

        for source_folder in os.listdir(emotion_path):
            source_path = os.path.join(emotion_path, source_folder)
            if not os.path.isdir(source_path): continue

            for fname in os.listdir(source_path):
                if not fname.lower().endswith('.wav'): continue
                fpath = os.path.join(source_path, fname)
                try:
                    emotion, gender = parse_eseravd(fname, emotion_folder)
                    if emotion is None or gender is None: continue
                    audio_data, sr = load_and_preprocess_audio(fpath)
                    if audio_data is None: continue
                    feat = extract_mfcc_zcr(audio_data, sr)
                    features_by_gender[gender].append(feat)
                    labels_by_gender[gender].append(emotion_map[emotion])
                    file_count[gender] += 1
                except Exception as e:
                    print(f'    Error ({fpath}): {e}')

    print(f'  Collected — Male: {file_count["male"]}, Female: {file_count["female"]}')
    process_dataset(features_by_gender, labels_by_gender, file_count, DATASET_NAME)

## 12. Proses Dataset IndoWaveSentiment (Indonesia)

**Catatan:** Dataset ini memiliki 5 emosi; `calm` di-skip karena tidak ada di label 6 kelas.  
Emosi yang digunakan: neutral, happy, sad, angry.

Format file: `ActorId-EmotionCode-GenderCode-Repetition.wav`

In [ ]:
DATASET_NAME   = 'IndoWaveSentiment'
indowave_path  = os.path.join(DATASET_ROOT, 'IndoWaveSentiment')

print(f"\n{'='*55}\nProcessing {DATASET_NAME} (Indonesia)...\n{'='*55}")

if not os.path.exists(indowave_path):
    print(f'[SKIP] Path tidak ditemukan: {indowave_path}')
else:
    features_by_gender = {'male': [], 'female': []}
    labels_by_gender   = {'male': [], 'female': []}
    file_count         = {'male': 0, 'female': 0}

    for actor_folder in sorted(os.listdir(indowave_path)):
        actor_path = os.path.join(indowave_path, actor_folder)
        if not os.path.isdir(actor_path): continue

        for fname in os.listdir(actor_path):
            if not fname.lower().endswith('.wav'): continue
            fpath = os.path.join(actor_path, fname)
            try:
                emotion, gender = parse_indowave(fname)
                if emotion is None or gender is None: continue
                audio_data, sr = load_and_preprocess_audio(fpath)
                if audio_data is None: continue
                feat = extract_mfcc_zcr(audio_data, sr)
                features_by_gender[gender].append(feat)
                labels_by_gender[gender].append(emotion_map[emotion])
                file_count[gender] += 1
            except Exception as e:
                print(f'  Error ({fpath}): {e}')

    print(f'  Collected — Male: {file_count["male"]}, Female: {file_count["female"]}')
    process_dataset(features_by_gender, labels_by_gender, file_count, DATASET_NAME)

## 13. Ringkasan Akhir Semua Dataset

In [ ]:
print('\n' + '='*60)
print('FEATURE EXTRACTION COMPLETED!')
print('='*60)
print(f'\nLabel sistem: {emotion_map}\n')

# Hitung total per gender
all_stats = {}
for gender in ['male', 'female']:
    gender_dir = os.path.join(OUTPUT_DIR, gender)
    if not os.path.exists(gender_dir): continue

    npz_files = sorted([f for f in os.listdir(gender_dir) if f.endswith('.npz')])
    print(f'\n{gender.capitalize()} dataset files:')
    print(f'  {"Dataset":<25} {"Samples":>10} {"Shape"}')
    print(f'  {"-"*55}')

    total_samples = 0
    combined_labels = []
    for fname in npz_files:
        fpath = os.path.join(gender_dir, fname)
        data  = np.load(fpath)
        n = data['features'].shape[0]
        total_samples += n
        combined_labels.extend(data['labels'].tolist())
        ds_name = fname.replace('_features.npz', '')
        print(f'  {ds_name:<25} {n:>10,}  {data["features"].shape}')

    print(f'  {"-"*55}')
    print(f'  {"TOTAL":<25} {total_samples:>10,}')

    if combined_labels:
        combined_labels = np.array(combined_labels)
        print(f'\n  Distribusi gabungan [{gender}]:')
        for lbl, cnt in zip(*np.unique(combined_labels, return_counts=True)):
            bar = '█' * int(cnt / max(np.unique(combined_labels, return_counts=True)[1]) * 30)
            print(f'    {inv_emotion_map[lbl]:<10}: {cnt:>6,}  {bar}')

print('\n' + '='*60)
print('File .npz siap untuk model_training_inline.ipynb')
print('='*60)

## 14. Penyeimbangan Data Gabungan (Undersampling)

Setelah semua dataset digabungkan, distribusi kelas emosi bisa tidak seimbang.  
Sel ini menerapkan **random undersampling** — memangkas kelas yang jumlahnya lebih banyak  
hingga sama dengan kelas yang jumlahnya **paling sedikit** (per gender).  

Output disimpan ke:  
- `extracted_features/combined_balanced_male.npz`  
- `extracted_features/combined_balanced_female.npz`

In [ ]:
import random

random.seed(42)
np.random.seed(42)


def undersample_combined(gender):
    """
    Gabungkan semua file .npz untuk gender tertentu,
    lalu lakukan random undersampling ke kelas terkecil.
    """
    gender_dir = os.path.join(OUTPUT_DIR, gender)
    if not os.path.exists(gender_dir):
        print(f'[SKIP] Folder tidak ditemukan: {gender_dir}')
        return None, None

    npz_files = sorted([f for f in os.listdir(gender_dir) if f.endswith('.npz')])
    if not npz_files:
        print(f'[SKIP] Tidak ada file .npz di {gender_dir}')
        return None, None

    all_features, all_labels = [], []
    for fname in npz_files:
        fpath = os.path.join(gender_dir, fname)
        data  = np.load(fpath)
        all_features.append(data['features'])
        all_labels.append(data['labels'])
        print(f'  Loaded {fname}: {data["features"].shape[0]:,} sampel')

    X = np.concatenate(all_features, axis=0)
    y = np.concatenate(all_labels,   axis=0)

    print(f'\n  Total gabungan [{gender}]: {len(y):,} sampel')
    print('  Distribusi SEBELUM undersampling:')
    unique_lbls, counts = np.unique(y, return_counts=True)
    class_counts = {}
    for lbl, cnt in zip(unique_lbls, counts):
        class_counts[int(lbl)] = int(cnt)
        bar = chr(9608) * int(cnt / max(counts) * 30)
        print(f'    {inv_emotion_map[lbl]:<10}: {cnt:>6,}  {bar}')

    # Kelas yang tidak ada sama sekali tidak dimasukkan ke min
    present_classes = {k: v for k, v in class_counts.items() if v > 0}
    min_count = min(present_classes.values())
    print(f'\n  Target sampel per kelas (min): {min_count:,}')

    # ── Random undersampling ─────────────────────────────────────────────────
    balanced_idx = []
    for lbl in unique_lbls:
        idx = np.where(y == lbl)[0].tolist()
        sampled = random.sample(idx, min_count)
        balanced_idx.extend(sampled)

    random.shuffle(balanced_idx)
    X_bal = X[balanced_idx]
    y_bal = y[balanced_idx]

    print('\n  Distribusi SETELAH undersampling:')
    for lbl, cnt in zip(*np.unique(y_bal, return_counts=True)):
        bar = chr(9608) * 30
        print(f'    {inv_emotion_map[lbl]:<10}: {cnt:>6,}  {bar}')

    # ── Simpan ───────────────────────────────────────────────────────────────
    out_path = os.path.join(OUTPUT_DIR, f'combined_balanced_{gender}.npz')
    np.savez_compressed(out_path, features=X_bal, labels=y_bal)
    print(f'\n  ✅ Saved {len(y_bal):,} sampel → {out_path}')
    return X_bal, y_bal


print('=' * 60)
print('PENYEIMBANGAN DATA GABUNGAN (UNDERSAMPLING)')
print('=' * 60)

for g in ['male', 'female']:
    print(f'\n{"-" * 55}')
    print(f'Gender: {g.upper()}')
    print(f'{"-" * 55}')
    undersample_combined(g)

# ── Visualisasi distribusi akhir ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribusi Kelas Emosi — Data Gabungan Setelah Undersampling',
             fontsize=13, fontweight='bold')

palette = sns.color_palette('Set2', len(emotion_map))
for ax, g in zip(axes, ['male', 'female']):
    out_path = os.path.join(OUTPUT_DIR, f'combined_balanced_{g}.npz')
    if not os.path.exists(out_path):
        ax.set_title(f'{g.capitalize()} (tidak ada data)')
        continue
    data = np.load(out_path)
    y_g  = data['labels']
    emo_names = [inv_emotion_map[i] for i in sorted(emotion_map.values())]
    cnts      = [int(np.sum(y_g == i)) for i in sorted(emotion_map.values())]
    bars = ax.bar(emo_names, cnts, color=palette)
    ax.set_title(f'{g.capitalize()} (Total: {sum(cnts):,})', fontsize=11)
    ax.set_xlabel('Emosi')
    ax.set_ylabel('Jumlah Sampel')
    ax.set_xticklabels(emo_names, rotation=30, ha='right')
    for bar, cnt in zip(bars, cnts):
        if cnt > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    str(cnt), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'combined_balanced_distribution.png')
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'\n✅ Plot distribusi disimpan: {save_path}')